In [ ]:
import tkinter as tk
from tkinter import ttk, messagebox, simpledialog
import random
import queue
import socket
import threading
from collections import defaultdict, deque
from datetime import datetime
import json
import os
import hashlib
import math


USERS_FILE = "users.json"
PROCESS_DIR = "process_files"
PAGE_SIZE = 4096  # 4KB
MEMORY_SIZE = 1024 * 1024 * 16  # 16MB


if not os.path.exists(PROCESS_DIR):
    os.makedirs(PROCESS_DIR)

def load_users():
    if os.path.exists(USERS_FILE):
        try:
            with open(USERS_FILE, "r") as f:
                return json.load(f)
        except Exception:
            return {}
    return {}

def save_users(users):
    with open(USERS_FILE, "w") as f:
        json.dump(users, f, indent=2)

def hash_password(password):
    """Hash a password using SHA-256 with salt"""
    salt = "os_simulation_salt"
    return hashlib.sha256((password + salt).encode()).hexdigest()

class PCB:
    def __init__(self, name, owner, priority, process_id, parent=None):
        self.process_id = process_id
        self.name = name
        self.state = "New"
        self.owner = owner
        self.priority = priority
        self.parent = parent
        self.children = []
        
        # Memory management
        self.memory_required = random.randint(1, 10) * PAGE_SIZE
        self.required_pages = math.ceil(self.memory_required / PAGE_SIZE)
        self.allocated_pages = []
        
        # CPU scheduling attributes
        self.arrival_time = datetime.now()
        self.burst_time = random.randint(2, 8)
        self.remaining_time = self.burst_time
        self.waiting_time = 0
        self.turnaround_time = 0
        

        self.registers = {
            "PC": random.randint(0, 1000),  
            "SP": random.randint(1000, 2000), 
            "IR": 0,   
            "AC": 0,  
            "X": random.randint(0, 100),
            "Y": random.randint(0, 100)
        }
        
        self.processor = "CPU-0"
        self.io_state = "No I/O"
        self.start_time = datetime.now()
        self.last_run_time = None
        
        if parent:
            parent.children.append(self)

class Semaphore:
    def __init__(self, name, value):
        self.name = name
        self.value = value
        self.queue = deque()

class Resource:
    def __init__(self, name, total):
        self.name = name
        self.total = total
        self.allocated = {}

class SocketServerThread(threading.Thread):
    def __init__(self, app, port=9000):
        super().__init__(daemon=True)
        self.app = app
        self.port = port
        self.running = True

    def run(self):
        s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        s.bind(('localhost', self.port))
        s.listen(1)
        self.app.socket_server_status = "Started"
        while self.running:
            try:
                s.settimeout(1.0)
                conn, addr = s.accept()
                data = conn.recv(1024)
                msg = data.decode()
                conn.sendall(f"Received: {msg}".encode())
                self.app.last_socket_msg = f"From {addr}: {msg}"
                conn.close()
            except socket.timeout:
                continue
            except Exception:
                break
        s.close()
        self.app.socket_server_status = "Stopped"

class OSSimulationApp:
    def __init__(self, root):
        self.root = root
        self.root.title("Advanced OS Simulation System")
        self.root.geometry("1100x780")
        self.root.config(bg="#e0c3fc")
        
        # Initialize system components
        self.users = load_users()
        self.current_user = None
        self.processes = {}
        self.ready_queue = deque()
        self.blocked_queue = deque()
        self.suspended_queue = deque()
        self.running_process = None
        self.scheduling_algorithms = ["FCFS", "SJF", "Priority", "RR"]
        self.scheduling_algorithm = "FCFS"
        self.time_quantum = 2
        self.page_table = {}
        self.lru_pages = {}  # {page_num: last_access_time}
        self.semaphores = {}
        self.shared_memory = {}
        self.message_queues = defaultdict(queue.Queue)
        
        # Resources
        self.resources = {
            "printer": Resource("printer", 2),
            "scanner": Resource("scanner", 1),
            "modem": Resource("modem", 1),
            "tape_drive": Resource("tape_drive", 2)
        }
        
        self.last_action_time = datetime.now()
        self.show_password_var = tk.BooleanVar()
        self.socket_server_thread = None
        self.socket_server_status = "Stopped"
        self.last_socket_msg = ""
        
        self.show_login_screen()
        self.root.bind("<Motion>", self.user_action)
        self.auto_logout_timer()

    def clear_window(self):
        for widget in self.root.winfo_children():
            widget.destroy()

    # -------------------- AUTHENTICATION --------------------
    def show_login_screen(self):
        self.clear_window()

        # Main container frame
        container = tk.Frame(self.root, bg="#e0c3fc")
        container.pack(fill="both", expand=True, padx=20, pady=20)

        # Card frame
        card = tk.Frame(container, bg="#f7f7fa", bd=2, relief=tk.RIDGE)
        card.place(relx=0.5, rely=0.5, anchor="center", width=400, height=520)

        # Logo/icon
        icon_frame = tk.Frame(card, bg="#f7f7fa")
        icon_frame.pack(pady=20)
        icon = tk.Label(icon_frame, text="⚙️", font=("Segoe UI", 48), bg="#f7f7fa")
        icon.pack()

        # Email field
        email_frame = tk.Frame(card, bg="#f7f7fa")
        email_frame.pack(pady=10, padx=30, fill="x")
        tk.Label(email_frame, text="Email:", font=("Segoe UI", 12), bg="#f7f7fa", anchor="w").pack(fill="x")
        self.email_entry = tk.Entry(email_frame, font=("Segoe UI", 12), bd=1, relief=tk.SOLID)
        self.email_entry.pack(fill="x", ipady=5)

        # Password field
        pass_frame = tk.Frame(card, bg="#f7f7fa")
        pass_frame.pack(pady=10, padx=30, fill="x")
        tk.Label(pass_frame, text="Password:", font=("Segoe UI", 12), bg="#f7f7fa", anchor="w").pack(fill="x")
        self.pass_entry = tk.Entry(pass_frame, font=("Segoe UI", 12), bd=1, relief=tk.SOLID, show="*")
        self.pass_entry.pack(fill="x", ipady=5)

        # Show password checkbox
        show_pass_cb = tk.Checkbutton(card, text="Show Password", font=("Segoe UI", 10),
                                     bg="#f7f7fa", variable=self.show_password_var,
                                     command=self.toggle_password_visibility)
        show_pass_cb.pack(pady=5)

        # Login button
        login_btn = tk.Button(card, text="LOGIN", font=("Segoe UI", 12, "bold"), bg="#23395d",
                             fg="white", height=2, width=20, command=self.login)
        login_btn.pack(pady=20)

        # Register link
        reg_frame = tk.Frame(card, bg="#f7f7fa")
        reg_frame.pack(pady=10)
        tk.Label(reg_frame, text="Don't have an account?", font=("Segoe UI", 10), bg="#f7f7fa").pack(side="left")
        reg_btn = tk.Button(reg_frame, text="Register", font=("Segoe UI", 10, "bold"),
                           bg="#f7f7fa", fg="#23395d", bd=0, command=self.show_register_screen)
        reg_btn.pack(side="left")

    def toggle_password_visibility(self):
        if self.show_password_var.get():
            self.pass_entry.config(show="")
        else:
            self.pass_entry.config(show="*")

    def login(self):
        email = self.email_entry.get().strip().lower()
        password = self.pass_entry.get()

        if not email or not password:
            messagebox.showerror("Error", "Please enter both email and password")
            return

        if email not in self.users:
            if messagebox.askyesno("Not Registered", "Email not found. Would you like to register?"):
                self.show_register_screen(email)
            return

        # Verify hashed password
        hashed_input = hash_password(password)
        if self.users[email]["password"] != hashed_input:
            messagebox.showerror("Error", "Incorrect password")
            return

        self.current_user = email
        self.show_main_panel()

    def show_register_screen(self, email=""):
        self.clear_window()
        self.show_register_password_var = tk.BooleanVar()

        # Main container frame
        container = tk.Frame(self.root, bg="#e0c3fc")
        container.pack(fill="both", expand=True, padx=20, pady=20)

        # Card frame
        card = tk.Frame(container, bg="#f7f7fa", bd=2, relief=tk.RIDGE)
        card.place(relx=0.5, rely=0.5, anchor="center", width=400, height=530)

        # Title
        tk.Label(card, text="Register New Account", font=("Segoe UI", 16, "bold"),
                bg="#f7f7fa", fg="#23395d").pack(pady=20)

        # Email field
        email_frame = tk.Frame(card, bg="#f7f7fa")
        email_frame.pack(pady=10, padx=30, fill="x")
        tk.Label(email_frame, text="Email:", font=("Segoe UI", 12), bg="#f7f7fa", anchor="w").pack(fill="x")
        email_entry = tk.Entry(email_frame, font=("Segoe UI", 12), bd=1, relief=tk.SOLID)
        email_entry.pack(fill="x", ipady=5)
        if email:
            email_entry.insert(0, email)

        # Password field
        pass_frame = tk.Frame(card, bg="#f7f7fa")
        pass_frame.pack(pady=10, padx=30, fill="x")
        tk.Label(pass_frame, text="Password:", font=("Segoe UI", 12), bg="#f7f7fa", anchor="w").pack(fill="x")
        pass_entry = tk.Entry(pass_frame, font=("Segoe UI", 12), bd=1, relief=tk.SOLID, show="*")
        pass_entry.pack(fill="x", ipady=5)

        # Confirm Password field
        confirm_frame = tk.Frame(card, bg="#f7f7fa")
        confirm_frame.pack(pady=10, padx=30, fill="x")
        tk.Label(confirm_frame, text="Confirm Password:", font=("Segoe UI", 12), bg="#f7f7fa", anchor="w").pack(fill="x")
        confirm_entry = tk.Entry(confirm_frame, font=("Segoe UI", 12), bd=1, relief=tk.SOLID, show="*")
        confirm_entry.pack(fill="x", ipady=5)

        # Show Password checkbox
        show_pass_cb = tk.Checkbutton(card, text="Show Password", font=("Segoe UI", 10),
                                     bg="#f7f7fa", variable=self.show_register_password_var,
                                     command=lambda: self.toggle_register_password_visibility(pass_entry, confirm_entry))
        show_pass_cb.pack(pady=5)

        def register():
            email_val = email_entry.get().strip().lower()
            pass_val = pass_entry.get()
            confirm_val = confirm_entry.get()

            if not email_val or not pass_val or not confirm_val:
                messagebox.showerror("Error", "All fields are required")
                return

            if pass_val != confirm_val:
                messagebox.showerror("Error", "Passwords do not match")
                return

            # Password restrictions
            if len(pass_val) < 8:
                messagebox.showerror("Error", "Password must be at least 8 characters long")
                return
            if not any(c.islower() for c in pass_val):
                messagebox.showerror("Error", "Password must contain at least one lowercase letter")
                return
            if not any(c.isupper() for c in pass_val):
                messagebox.showerror("Error", "Password must contain at least one uppercase letter")
                return
            if not any(c.isdigit() for c in pass_val):
                messagebox.showerror("Error", "Password must contain at least one digit")
                return
            if not any(c in "!@#$%^&*()-_=+[]{}|;:'\",.<>?/`~" for c in pass_val):
                messagebox.showerror("Error", "Password must contain at least one special character")
                return

            if email_val in self.users:
                messagebox.showerror("Error", "Email already registered")
                return

            # Store hashed password
            self.users[email_val] = {"password": hash_password(pass_val)}
            save_users(self.users)
            messagebox.showinfo("Success", "Registration successful!")
            self.show_login_screen()

        # Register button
        reg_btn = tk.Button(card, text="REGISTER", font=("Segoe UI", 12, "bold"), bg="#23395d",
                           fg="white", height=2, width=20, command=register)
        reg_btn.pack(pady=20)

        # Back to login
        back_btn = tk.Button(card, text="Back to Login", font=("Segoe UI", 10),
                            bg="#f7f7fa", fg="#23395d", bd=0, command=self.show_login_screen)
        back_btn.pack()

    def toggle_register_password_visibility(self, pass_entry, confirm_entry):
        if self.show_register_password_var.get():
            pass_entry.config(show="")
            confirm_entry.config(show="")
        else:
            pass_entry.config(show="*")
            confirm_entry.config(show="*")

    def logout(self):
        self.current_user = None
        self.show_login_screen()

    def user_action(self, event=None):
        self.last_action_time = datetime.now()

    def auto_logout_timer(self):
        if self.current_user and (datetime.now() - self.last_action_time).seconds > 240:
            self.current_user = None
            messagebox.showinfo("Logged Out", "You were logged out due to inactivity")
            self.show_login_screen()
        self.root.after(5000, self.auto_logout_timer)

    # -------------------- PROCESS MANAGEMENT --------------------
    def show_main_panel(self):
        self.clear_window()

        # Header frame
        header = tk.Frame(self.root, bg="#23395d")
        header.pack(fill="x")

        tk.Label(header, text=f"OS Simulation - {self.current_user}", font=("Segoe UI", 16, "bold"),
                bg="#23395d", fg="white").pack(side="left", padx=20, pady=10)

        logout_btn = tk.Button(header, text="Logout", font=("Segoe UI", 10),
                              bg="#b7b1f3", fg="#23395d", command=self.logout)
        logout_btn.pack(side="right", padx=20, pady=10)

        # Main content frame
        content = tk.Frame(self.root, bg="#f7f7fa")
        content.pack(fill="both", expand=True, padx=20, pady=20)

        # Button grid
        buttons = [
            ("Process Management", self.show_process_management),
            ("Memory Management", self.show_memory_management),
            ("I/O Management", self.show_io_management),
            ("Synchronization", self.show_sync_operations),
            ("Configuration", self.show_configuration)
        ]

        for i, (text, cmd) in enumerate(buttons):
            btn = tk.Button(content, text=text, command=cmd,
                           font=("Segoe UI", 12, "bold"), bg="#23395d", fg="white",
                           height=3, width=20)
            btn.grid(row=i // 2, column=i % 2, padx=10, pady=10, sticky="nsew")
            content.grid_columnconfigure(i % 2, weight=1)

        content.grid_rowconfigure(0, weight=1)
        content.grid_rowconfigure(1, weight=1)
        content.grid_rowconfigure(2, weight=1)

    def show_process_management(self):
        self.clear_window()

        # Header
        header = tk.Frame(self.root, bg="#23395d")
        header.pack(fill="x")

        tk.Label(header, text="Process Management", font=("Segoe UI", 16, "bold"),
                bg="#23395d", fg="white").pack(side="left", padx=20, pady=10)

        back_btn = tk.Button(header, text="Back", font=("Segoe UI", 10),
                            bg="#b7b1f3", fg="#23395d", command=self.show_main_panel)
        back_btn.pack(side="right", padx=20, pady=10)

        # Content frame
        content = tk.Frame(self.root, bg="#f7f7fa")
        content.pack(fill="both", expand=True, padx=20, pady=20)

        # Button grid
        buttons = [
            ("Create Process", self.create_process_dialog),
            ("Terminate Process", self.terminate_process_dialog),
            ("Block/Unblock Process", self.block_unblock_dialog),
            ("Suspend/Resume Process", self.suspend_resume_dialog),
            ("Schedule Process", self.dispatch_process_dialog),
            ("Change Priority", self.change_priority_dialog),
            ("Run Scheduler", self.schedule_next_process),
            ("View All Processes", lambda: self.show_process_table(parent=content)),
        ]

        for i, (text, cmd) in enumerate(buttons):
            btn = tk.Button(content, text=text, command=cmd,
                           font=("Segoe UI", 11), bg="#23395d", fg="white",
                           height=2, width=20, wraplength=150)
            btn.grid(row=i // 3, column=i % 3, padx=5, pady=5, sticky="nsew")
            content.grid_columnconfigure(i % 3, weight=1)

        for i in range(3):
            content.grid_rowconfigure(i, weight=1)

        # Show process table at bottom
        self.show_process_table(parent=content)

    def create_process_file(self, pcb):
        """Create a JSON file with process details"""
        process_data = {
            "pid": pcb.process_id,
            "name": pcb.name,
            "owner": pcb.owner,
            "priority": pcb.priority,
            "parent": pcb.parent.process_id if pcb.parent else None,
            "state": pcb.state,
            "memory_required": pcb.memory_required,
            "required_pages": pcb.required_pages,
            "allocated_pages": pcb.allocated_pages,
            "burst_time": pcb.burst_time,
            "remaining_time": pcb.remaining_time,
            "arrival_time": pcb.arrival_time.isoformat(),
            "start_time": pcb.start_time.isoformat(),
            "registers": pcb.registers,
            "io_state": pcb.io_state
        }
        
        filename = os.path.join(PROCESS_DIR, f"process_{pcb.process_id}.json")
        with open(filename, "w") as f:
            json.dump(process_data, f, indent=2, default=str)

    def create_process_dialog(self):
        dialog = tk.Toplevel(self.root)
        dialog.title("Create New Process")
        dialog.configure(bg="#f7f7fa")

        tk.Label(dialog, text="Process ID:", font=("Segoe UI", 12), bg="#f7f7fa").pack(pady=5)
        id_entry = tk.Entry(dialog, font=("Segoe UI", 12), width=20)
        id_entry.pack(pady=5)

        tk.Label(dialog, text="Process Name:", font=("Segoe UI", 12), bg="#f7f7fa").pack(pady=5)
        name_entry = tk.Entry(dialog, font=("Segoe UI", 12), width=20)
        name_entry.pack(pady=5)

        tk.Label(dialog, text="Priority (1-10):", font=("Segoe UI", 12), bg="#f7f7fa").pack(pady=5)
        prio_entry = tk.Entry(dialog, font=("Segoe UI", 12), width=20)
        prio_entry.insert(0, "5")
        prio_entry.pack(pady=5)

        tk.Label(dialog, text="Burst Time (1-10):", font=("Segoe UI", 12), bg="#f7f7fa").pack(pady=5)
        burst_entry = tk.Entry(dialog, font=("Segoe UI", 12), width=20)
        burst_entry.insert(0, str(random.randint(2, 8)))
        burst_entry.pack(pady=5)

        tk.Label(dialog, text="Parent ID (optional):", font=("Segoe UI", 12), bg="#f7f7fa").pack(pady=5)
        parent_entry = tk.Entry(dialog, font=("Segoe UI", 12), width=20)
        parent_entry.pack(pady=5)

        def create():
            try:
                pid = int(id_entry.get())
                if pid in self.processes:
                    messagebox.showerror("Error", "Process ID already exists")
                    return
            except ValueError:
                messagebox.showerror("Error", "Process ID must be a number")
                return

            name = name_entry.get().strip()
            if not name:
                messagebox.showerror("Error", "Process name required")
                return

            try:
                priority = int(prio_entry.get())
                if not 1 <= priority <= 10:
                    raise ValueError()
            except ValueError:
                messagebox.showerror("Error", "Priority must be between 1-10")
                return

            try:
                burst_time = int(burst_entry.get())
                if burst_time <= 0:
                    raise ValueError()
            except ValueError:
                messagebox.showerror("Error", "Burst time must be positive integer")
                return

            parent_id = parent_entry.get().strip()
            parent = None
            if parent_id:
                try:
                    parent_id = int(parent_id)
                    if parent_id in self.processes:
                        parent = self.processes[parent_id]
                    else:
                        messagebox.showerror("Error", "Parent process not found")
                        return
                except ValueError:
                    messagebox.showerror("Error", "Invalid parent ID")
                    return

            p = PCB(name, self.current_user, priority, pid, parent)
            p.burst_time = burst_time
            p.remaining_time = burst_time
            
            self.processes[pid] = p
            self.ready_queue.append(p)
            p.state = "Ready"
            
            # Create process file
            self.create_process_file(p)
            
            dialog.destroy()
            self.show_process_management()

        tk.Button(dialog, text="Create", command=create, font=("Segoe UI", 12, "bold"),
                 bg="#23395d", fg="#fff", width=15).pack(side="left", padx=10, pady=10)
        tk.Button(dialog, text="Cancel", command=dialog.destroy, font=("Segoe UI", 12, "bold"),
                 bg="#b7b1f3", fg="#fff", width=15).pack(side="right", padx=10, pady=10)

    def terminate_process(self, pid):
        """Terminate a process by ID"""
        if pid not in self.processes:
            return False

        p = self.processes[pid]

        # Terminate all children recursively
        def terminate_children(process):
            for child in process.children:
                if child.process_id in self.processes:
                    terminate_children(child)
                    
                    # Remove process file
                    filename = os.path.join(PROCESS_DIR, f"process_{child.process_id}.json")
                    if os.path.exists(filename):
                        os.remove(filename)
                        
                    del self.processes[child.process_id]

        terminate_children(p)

        # Remove from queues
        if p in self.ready_queue:
            self.ready_queue.remove(p)
        if p in self.blocked_queue:
            self.blocked_queue.remove(p)
        if p in self.suspended_queue:
            self.suspended_queue.remove(p)

        # Remove from memory
        if pid in self.page_table:
            for page in self.page_table[pid]:
                if page in self.lru_pages:
                    del self.lru_pages[page]
            del self.page_table[pid]

        # Remove process file
        filename = os.path.join(PROCESS_DIR, f"process_{pid}.json")
        if os.path.exists(filename):
            os.remove(filename)

        # Remove the process
        del self.processes[pid]
        
        # If terminating the running process
        if self.running_process and self.running_process.process_id == pid:
            self.running_process = None
            
        return True

    def terminate_process_dialog(self):
        dialog = tk.Toplevel(self.root)
        dialog.title("Terminate Process")
        dialog.configure(bg="#f7f7fa")

        tk.Label(dialog, text="Process ID to terminate:", font=("Segoe UI", 12), bg="#f7f7fa").pack(pady=10)
        pid_entry = tk.Entry(dialog, font=("Segoe UI", 12), width=20)
        pid_entry.pack(pady=5)

        def terminate():
            try:
                pid = int(pid_entry.get())
                if self.terminate_process(pid):
                    dialog.destroy()
                    self.show_process_management()
                else:
                    messagebox.showerror("Error", "Process not found")
            except ValueError:
                messagebox.showerror("Error", "Invalid process ID")

        tk.Button(dialog, text="Terminate", command=terminate, font=("Segoe UI", 12, "bold"),
                 bg="#23395d", fg="#fff", width=15).pack(side="left", padx=10, pady=10)
        tk.Button(dialog, text="Cancel", command=dialog.destroy, font=("Segoe UI", 12, "bold"),
                 bg="#b7b1f3", fg="#fff", width=15).pack(side="right", padx=10, pady=10)

    def block_unblock_dialog(self):
        dialog = tk.Toplevel(self.root)
        dialog.title("Block/Unblock Process")
        dialog.configure(bg="#f7f7fa")

        tk.Label(dialog, text="Process ID:", font=("Segoe UI", 12), bg="#f7f7fa").pack(pady=10)
        pid_entry = tk.Entry(dialog, font=("Segoe UI", 12), width=20)
        pid_entry.pack(pady=5)

        def block():
            try:
                pid = int(pid_entry.get())
                if pid not in self.processes:
                    messagebox.showerror("Error", "Process not found")
                    return

                p = self.processes[pid]
                if p.state == "Blocked":
                    messagebox.showerror("Error", "Process already blocked")
                    return

                p.state = "Blocked"
                self.blocked_queue.append(p)
                if p in self.ready_queue:
                    self.ready_queue.remove(p)
                if self.running_process and self.running_process.process_id == pid:
                    self.running_process = None

                # Update process file
                self.create_process_file(p)
                
                dialog.destroy()
                self.show_process_management()
            except ValueError:
                messagebox.showerror("Error", "Invalid process ID")

        def unblock():
            try:
                pid = int(pid_entry.get())
                if pid not in self.processes:
                    messagebox.showerror("Error", "Process not found")
                    return

                p = self.processes[pid]
                if p.state != "Blocked":
                    messagebox.showerror("Error", "Process is not blocked")
                    return

                p.state = "Ready"
                if p in self.blocked_queue:
                    self.blocked_queue.remove(p)
                self.ready_queue.append(p)
                
                # Update process file
                self.create_process_file(p)
                
                dialog.destroy()
                self.show_process_management()
            except ValueError:
                messagebox.showerror("Error", "Invalid process ID")

        btn_frame = tk.Frame(dialog, bg="#f7f7fa")
        btn_frame.pack(pady=10)

        tk.Button(btn_frame, text="Block", command=block, font=("Segoe UI", 12, "bold"),
                 bg="#23395d", fg="#fff", width=10).pack(side="left", padx=5)
        tk.Button(btn_frame, text="Unblock", command=unblock, font=("Segoe UI", 12, "bold"),
                 bg="#e0c3fc", fg="#23395d", width=10).pack(side="left", padx=5)
        tk.Button(dialog, text="Cancel", command=dialog.destroy, font=("Segoe UI", 12, "bold"),
                 bg="#b7b1f3", fg="#fff", width=15).pack(pady=10)

    def suspend_resume_dialog(self):
        dialog = tk.Toplevel(self.root)
        dialog.title("Suspend/Resume Process")
        dialog.configure(bg="#f7f7fa")

        tk.Label(dialog, text="Process ID:", font=("Segoe UI", 12), bg="#f7f7fa").pack(pady=10)
        pid_entry = tk.Entry(dialog, font=("Segoe UI", 12), width=20)
        pid_entry.pack(pady=5)

        def suspend():
            try:
                pid = int(pid_entry.get())
                if pid not in self.processes:
                    messagebox.showerror("Error", "Process not found")
                    return

                p = self.processes[pid]
                if p.state == "Suspended":
                    messagebox.showerror("Error", "Process already suspended")
                    return

                p.state = "Suspended"
                self.suspended_queue.append(p)
                if p in self.ready_queue:
                    self.ready_queue.remove(p)
                if p in self.blocked_queue:
                    self.blocked_queue.remove(p)
                if self.running_process and self.running_process.process_id == pid:
                    self.running_process = None
                    
                # Update process file
                self.create_process_file(p)
                
                dialog.destroy()
                self.show_process_management()
            except ValueError:
                messagebox.showerror("Error", "Invalid process ID")

        def resume():
            try:
                pid = int(pid_entry.get())
                if pid not in self.processes:
                    messagebox.showerror("Error", "Process not found")
                    return

                p = self.processes[pid]
                if p.state != "Suspended":
                    messagebox.showerror("Error", "Process is not suspended")
                    return

                p.state = "Ready"
                if p in self.suspended_queue:
                    self.suspended_queue.remove(p)
                self.ready_queue.append(p)
                
                # Update process file
                self.create_process_file(p)
                
                dialog.destroy()
                self.show_process_management()
            except ValueError:
                messagebox.showerror("Error", "Invalid process ID")

        btn_frame = tk.Frame(dialog, bg="#f7f7fa")
        btn_frame.pack(pady=10)

        tk.Button(btn_frame, text="Suspend", command=suspend, font=("Segoe UI", 12, "bold"),
                 bg="#23395d", fg="#fff", width=10).pack(side="left", padx=5)
        tk.Button(btn_frame, text="Resume", command=resume, font=("Segoe UI", 12, "bold"),
                 bg="#e0c3fc", fg="#23395d", width=10).pack(side="left", padx=5)
        tk.Button(dialog, text="Cancel", command=dialog.destroy, font=("Segoe UI", 12, "bold"),
                 bg="#b7b1f3", fg="#fff", width=15).pack(pady=10)

    def dispatch_process_dialog(self):
        dialog = tk.Toplevel(self.root)
        dialog.title("Dispatch Process")
        dialog.configure(bg="#f7f7fa")

        tk.Label(dialog, text="Process ID to dispatch:", font=("Segoe UI", 12), bg="#f7f7fa").pack(pady=10)
        pid_entry = tk.Entry(dialog, font=("Segoe UI", 12), width=20)
        pid_entry.pack(pady=5)

        def dispatch():
            try:
                pid = int(pid_entry.get())
                if pid not in self.processes:
                    messagebox.showerror("Error", "Process not found")
                    return

                p = self.processes[pid]
                if p.state == "Running":
                    messagebox.showerror("Error", "Process already running")
                    return

                if self.running_process:
                    self.running_process.state = "Ready"
                    self.ready_queue.append(self.running_process)
                    # Update process file for previous running process
                    self.create_process_file(self.running_process)

                p.state = "Running"
                self.running_process = p
                p.last_run_time = datetime.now()
                
                # Update process file
                self.create_process_file(p)
                
                dialog.destroy()
                self.show_process_management()
            except ValueError:
                messagebox.showerror("Error", "Invalid process ID")

        tk.Button(dialog, text="Dispatch", command=dispatch, font=("Segoe UI", 12, "bold"),
                 bg="#23395d", fg="#fff", width=15).pack(side="left", padx=10, pady=10)
        tk.Button(dialog, text="Cancel", command=dialog.destroy, font=("Segoe UI", 12, "bold"),
                 bg="#b7b1f3", fg="#fff", width=15).pack(side="right", padx=10, pady=10)

    def change_priority_dialog(self):
        dialog = tk.Toplevel(self.root)
        dialog.title("Change Process Priority")
        dialog.configure(bg="#f7f7fa")

        tk.Label(dialog, text="Process ID:", font=("Segoe UI", 12), bg="#f7f7fa").pack(pady=10)
        pid_entry = tk.Entry(dialog, font=("Segoe UI", 12), width=20)
        pid_entry.pack(pady=5)

        tk.Label(dialog, text="New Priority (1-10):", font=("Segoe UI", 12), bg="#f7f7fa").pack(pady=10)
        prio_entry = tk.Entry(dialog, font=("Segoe UI", 12), width=20)
        prio_entry.pack(pady=5)

        def change():
            try:
                pid = int(pid_entry.get())
                if pid not in self.processes:
                    messagebox.showerror("Error", "Process not found")
                    return

                try:
                    new_prio = int(prio_entry.get())
                    if not 1 <= new_prio <= 10:
                        raise ValueError()
                except ValueError:
                    messagebox.showerror("Error", "Priority must be between 1-10")
                    return

                self.processes[pid].priority = new_prio
                
                # Update process file
                self.create_process_file(self.processes[pid])
                
                dialog.destroy()
                self.show_process_management()
            except ValueError:
                messagebox.showerror("Error", "Invalid process ID")

        tk.Button(dialog, text="Change", command=change, font=("Segoe UI", 12, "bold"),
                 bg="#23395d", fg="#fff", width=15).pack(side="left", padx=10, pady=10)
        tk.Button(dialog, text="Cancel", command=dialog.destroy, font=("Segoe UI", 12, "bold"),
                 bg="#b7b1f3", fg="#fff", width=15).pack(side="right", padx=10, pady=10)

    def schedule_next_process(self):
        if not self.ready_queue:
            messagebox.showinfo("Scheduling", "No processes in ready queue")
            return

        # Calculate waiting times
        current_time = datetime.now()
        for p in self.ready_queue:
            if p.last_run_time:
                p.waiting_time += (current_time - p.last_run_time).total_seconds()
            else:
                p.waiting_time += (current_time - p.arrival_time).total_seconds()

        if self.scheduling_algorithm == "FCFS":
            # First Come First Serve - select oldest process
            ready_queue_sorted = sorted(self.ready_queue, key=lambda p: p.arrival_time)
            next_process = ready_queue_sorted[0]
            algorithm_name = "First Come First Serve"

        elif self.scheduling_algorithm == "SJF":
            # Shortest Job First - select process with least remaining time
            ready_queue_sorted = sorted(self.ready_queue, key=lambda p: p.remaining_time)
            next_process = ready_queue_sorted[0]
            algorithm_name = "Shortest Job First"

        elif self.scheduling_algorithm == "Priority":
            # Priority Scheduling - select process with highest priority (lowest number)
            ready_queue_sorted = sorted(self.ready_queue, key=lambda p: p.priority)
            next_process = ready_queue_sorted[0]
            algorithm_name = "Priority Scheduling"

        elif self.scheduling_algorithm == "RR":
            # Round Robin - take from front of queue
            next_process = self.ready_queue.popleft()
            algorithm_name = f"Round Robin (Time Quantum: {self.time_quantum})"
        else:
            messagebox.showerror("Error", "Unknown scheduling algorithm")
            return

        # Handle currently running process
        if self.running_process:
            self.running_process.state = "Ready"
            self.ready_queue.append(self.running_process)
            # Update process file for previous running process
            self.create_process_file(self.running_process)

        # Assign CPU to next process
        next_process.state = "Running"
        self.running_process = next_process
        next_process.last_run_time = datetime.now()

        # Determine time slice
        if self.scheduling_algorithm == "RR":
            time_slice = min(self.time_quantum, next_process.remaining_time)
        else:
            time_slice = next_process.remaining_time

        # Update process statistics
        next_process.remaining_time -= time_slice
        if next_process.remaining_time < 0:
            next_process.remaining_time = 0

        # Calculate turnaround time
        next_process.turnaround_time = (datetime.now() - next_process.arrival_time).total_seconds()

        # Check if process completed
        if next_process.remaining_time <= 0:
            # Process completed
            messagebox.showinfo("Scheduling",
                               f"{algorithm_name} scheduled PID={next_process.process_id} (completed)\n"
                               f"Process {next_process.name} has finished execution\n"
                               f"Turnaround Time: {next_process.turnaround_time:.2f}s\n"
                               f"Waiting Time: {next_process.waiting_time:.2f}s")

            # Clean up completed process
            self.terminate_process(next_process.process_id)
        else:
            if self.scheduling_algorithm == "RR":
                # For Round Robin, put back in queue if not finished
                next_process.state = "Ready"
                self.ready_queue.append(next_process)
                self.running_process = None

            messagebox.showinfo("Scheduling",
                               f"{algorithm_name} scheduled PID={next_process.process_id}\n"
                               f"Process {next_process.name} executed for {time_slice} units\n"
                               f"Remaining time: {next_process.remaining_time}\n"
                               f"Turnaround Time: {next_process.turnaround_time:.2f}s\n"
                               f"Waiting Time: {next_process.waiting_time:.2f}s")

        # Update process file
        if next_process.process_id in self.processes:
            self.create_process_file(next_process)
            
        self.show_process_management()

    def show_process_table(self, parent=None):
        frame = tk.Frame(parent if parent else self.root, bg="#f7f7fa")
        frame.grid(row=4, column=0, columnspan=3, sticky="nsew", padx=10, pady=10)

        cols = ("PID", "Name", "State", "Priority", "Burst", "Remaining", "Memory", "Pages", "CPU", "I/O")

        tree = ttk.Treeview(frame, columns=cols, show="headings", height=10)
        vsb = ttk.Scrollbar(frame, orient="vertical", command=tree.yview)
        hsb = ttk.Scrollbar(frame, orient="horizontal", command=tree.xview)
        tree.configure(yscrollcommand=vsb.set, xscrollcommand=hsb.set)

        for col in cols:
            tree.heading(col, text=col)
            tree.column(col, width=100, anchor=tk.CENTER)

        for pcb in self.processes.values():
            tree.insert("", "end", values=(
                pcb.process_id,
                pcb.name,
                pcb.state,
                pcb.priority,
                pcb.burst_time,
                pcb.remaining_time,
                f"{pcb.memory_required//1024}KB",
                f"{len(pcb.allocated_pages)}/{pcb.required_pages}",
                pcb.processor,
                pcb.io_state
            ))

        tree.grid(row=0, column=0, sticky="nsew")
        vsb.grid(row=0, column=1, sticky="ns")
        hsb.grid(row=1, column=0, sticky="ew")
        frame.grid_rowconfigure(0, weight=1)
        frame.grid_columnconfigure(0, weight=1)

    # -------------------- MEMORY MANAGEMENT --------------------
    def show_memory_management(self):
        self.clear_window()

        # Header
        header = tk.Frame(self.root, bg="#23395d")
        header.pack(fill="x")

        tk.Label(header, text="Memory Management", font=("Segoe UI", 16, "bold"),
                bg="#23395d", fg="white").pack(side="left", padx=20, pady=10)

        back_btn = tk.Button(header, text="Back", font=("Segoe UI", 10),
                            bg="#b7b1f3", fg="#23395d", command=self.show_main_panel)
        back_btn.pack(side="right", padx=20, pady=10)

        # Content frame
        content = tk.Frame(self.root, bg="#f7f7fa")
        content.pack(fill="both", expand=True, padx=20, pady=20)

        # Button grid
        buttons = [
            ("Allocate Memory", self.allocate_memory_dialog),
            ("Free Memory", self.free_memory_dialog),
            ("LRU Page Replacement", self.lru_replace_dialog),
            ("View Memory Status", self.memory_status_dialog)
        ]

        for i, (text, cmd) in enumerate(buttons):
            btn = tk.Button(content, text=text, command=cmd,
                           font=("Segoe UI", 11), bg="#23395d", fg="white",
                           height=2, width=20, wraplength=150)
            btn.grid(row=i // 2, column=i % 2, padx=5, pady=5, sticky="nsew")
            content.grid_columnconfigure(i % 2, weight=1)

        for i in range(2):
            content.grid_rowconfigure(i, weight=1)

        # Show memory table at bottom
        self.show_memory_table(parent=content)

    def allocate_memory_dialog(self):
        dialog = tk.Toplevel(self.root)
        dialog.title("Allocate Memory")
        dialog.configure(bg="#f7f7fa")

        tk.Label(dialog, text="Process ID:", font=("Segoe UI", 12), bg="#f7f7fa").pack(pady=10)
        pid_entry = tk.Entry(dialog, font=("Segoe UI", 12), width=20)
        pid_entry.pack(pady=5)

        def allocate():
            try:
                pid = int(pid_entry.get())
                if pid not in self.processes:
                    messagebox.showerror("Error", "Process not found")
                    return

                p = self.processes[pid]
                if p.allocated_pages:
                    messagebox.showerror("Error", "Memory already allocated")
                    return

                # Find free pages
                used_pages = set()
                for pages in self.page_table.values():
                    used_pages.update(pages)

                free_pages = [i for i in range(MEMORY_SIZE // PAGE_SIZE) if i not in used_pages]

                if len(free_pages) < p.required_pages:
                    messagebox.showerror("Error", f"Not enough free memory (needed: {p.required_pages} pages)")
                    return

                # Allocate pages
                allocated_pages = free_pages[:p.required_pages]
                self.page_table[pid] = allocated_pages
                p.allocated_pages = allocated_pages

                # Update LRU tracking
                current_time = datetime.now()
                for page in allocated_pages:
                    self.lru_pages[page] = current_time

                # Update process file
                self.create_process_file(p)
                
                dialog.destroy()
                self.show_memory_management()
            except ValueError:
                messagebox.showerror("Error", "Invalid process ID")

        tk.Button(dialog, text="Allocate", command=allocate, font=("Segoe UI", 12, "bold"),
                 bg="#23395d", fg="#fff", width=15).pack(side="left", padx=10, pady=10)
        tk.Button(dialog, text="Cancel", command=dialog.destroy, font=("Segoe UI", 12, "bold"),
                 bg="#b7b1f3", fg="#fff", width=15).pack(side="right", padx=10, pady=10)

    def free_memory_dialog(self):
        dialog = tk.Toplevel(self.root)
        dialog.title("Free Memory")
        dialog.configure(bg="#f7f7fa")

        tk.Label(dialog, text="Process ID:", font=("Segoe UI", 12), bg="#f7f7fa").pack(pady=10)
        pid_entry = tk.Entry(dialog, font=("Segoe UI", 12), width=20)
        pid_entry.pack(pady=5)

        def free():
            try:
                pid = int(pid_entry.get())
                if pid not in self.processes:
                    messagebox.showerror("Error", "Process not found")
                    return

                p = self.processes[pid]
                if not p.allocated_pages:
                    messagebox.showerror("Error", "No memory allocated for this process")
                    return

                if pid in self.page_table:
                    # Remove from LRU tracking
                    for page in self.page_table[pid]:
                        if page in self.lru_pages:
                            del self.lru_pages[page]
                    del self.page_table[pid]

                p.allocated_pages = []
                
                # Update process file
                self.create_process_file(p)
                
                dialog.destroy()
                self.show_memory_management()
            except ValueError:
                messagebox.showerror("Error", "Invalid process ID")

        tk.Button(dialog, text="Free", command=free, font=("Segoe UI", 12, "bold"),
                 bg="#23395d", fg="#fff", width=15).pack(side="left", padx=10, pady=10)
        tk.Button(dialog, text="Cancel", command=dialog.destroy, font=("Segoe UI", 12, "bold"),
                 bg="#b7b1f3", fg="#fff", width=15).pack(side="right", padx=10, pady=10)

    def lru_replace_dialog(self):
        if not self.lru_pages:
            messagebox.showinfo("LRU", "No pages allocated to replace")
            return

        # Find least recently used page
        lru_page = min(self.lru_pages.keys(), key=lambda k: self.lru_pages[k])
        pid = None
        
        # Find which process owns this page
        for proc_id, pages in self.page_table.items():
            if lru_page in pages:
                pid = proc_id
                break

        if pid is None:
            messagebox.showerror("Error", "Could not find process for page")
            return

        # Remove from process's page table
        if pid in self.page_table and lru_page in self.page_table[pid]:
            self.page_table[pid].remove(lru_page)
            if not self.page_table[pid]:  # No more pages allocated
                self.processes[pid].allocated_pages = []
                del self.page_table[pid]

        # Remove from LRU tracking
        del self.lru_pages[lru_page]

        messagebox.showinfo("LRU Page Replacement",
                          f"Page {lru_page} from process {pid} was replaced\n"
                          f"(Least Recently Used page replacement)")

        # Update process file
        if pid in self.processes:
            self.create_process_file(self.processes[pid])
            
        self.show_memory_management()

    def memory_status_dialog(self):
        dialog = tk.Toplevel(self.root)
        dialog.title("Memory Status")
        dialog.configure(bg="#f7f7fa")

        used_pages = sum(len(pages) for pages in self.page_table.values())
        total_pages = MEMORY_SIZE // PAGE_SIZE

        tk.Label(dialog, text=f"Memory Usage: {used_pages}/{total_pages} pages",
                 font=("Segoe UI", 14), bg="#f7f7fa").pack(pady=10)

        tk.Label(dialog, text=f"Page Size: {PAGE_SIZE // 1024}KB",
                 font=("Segoe UI", 12), bg="#f7f7fa").pack(pady=5)

        tk.Label(dialog, text=f"Total Memory: {MEMORY_SIZE // (1024 * 1024)}MB",
                 font=("Segoe UI", 12), bg="#f7f7fa").pack(pady=5)

        tk.Label(dialog, text="Allocated Pages:",
                 font=("Segoe UI", 12, "bold"), bg="#f7f7fa").pack(pady=10)

        frame = tk.Frame(dialog, bg="#f7f7fa")
        frame.pack(fill="both", expand=True, padx=10, pady=5)

        text = tk.Text(frame, height=10, width=50, font=("Segoe UI", 10))
        scroll = tk.Scrollbar(frame, command=text.yview)
        text.configure(yscrollcommand=scroll.set)

        for pid, pages in self.page_table.items():
            p = self.processes.get(pid, None)
            pname = p.name if p else "Unknown"
            text.insert(tk.END, f"Process {pid} ({pname}): {len(pages)} pages\n")
            text.insert(tk.END, f"  Page→Frame: {', '.join(f'{p}→{p}' for p in pages)}\n\n")

        text.pack(side="left", fill="both", expand=True)
        scroll.pack(side="right", fill="y")

        tk.Button(dialog, text="Close", command=dialog.destroy, font=("Segoe UI", 12, "bold"),
                 bg="#b7b1f3", fg="#fff", width=15).pack(pady=10)

    def show_memory_table(self, parent):
        frame = tk.Frame(parent, bg="#f7f7fa")
        frame.grid(row=2, column=0, columnspan=2, sticky="nsew", padx=10, pady=10)

        cols = ("ProcessID", "Name", "MemReq(KB)", "Allocated", "Pages", "Page→Frame")

        tree = ttk.Treeview(frame, columns=cols, show="headings", height=10)
        vsb = ttk.Scrollbar(frame, orient="vertical", command=tree.yview)
        hsb = ttk.Scrollbar(frame, orient="horizontal", command=tree.xview)
        tree.configure(yscrollcommand=vsb.set, xscrollcommand=hsb.set)

        for col in cols:
            tree.heading(col, text=col)
            tree.column(col, width=120, anchor=tk.CENTER)

        for p in self.processes.values():
            alloc = "Yes" if p.allocated_pages else "No"
            pages = self.page_table.get(p.process_id, [])
            page_frames = ", ".join(f"{p}→{p}" for p in pages) if pages else "-"
            tree.insert("", "end", values=(
                p.process_id, p.name, p.memory_required // 1024,
                alloc, f"{len(pages)}/{p.required_pages}", page_frames
            ))

        tree.grid(row=0, column=0, sticky="nsew")
        vsb.grid(row=0, column=1, sticky="ns")
        hsb.grid(row=1, column=0, sticky="ew")
        frame.grid_rowconfigure(0, weight=1)
        frame.grid_columnconfigure(0, weight=1)

    # -------------------- I/O MANAGEMENT --------------------
    def show_io_management(self):
        self.clear_window()

        # Header
        header = tk.Frame(self.root, bg="#23395d")
        header.pack(fill="x")

        tk.Label(header, text="I/O Management", font=("Segoe UI", 16, "bold"),
                bg="#23395d", fg="white").pack(side="left", padx=20, pady=10)

        back_btn = tk.Button(header, text="Back", font=("Segoe UI", 10),
                            bg="#b7b1f3", fg="#23395d", command=self.show_main_panel)
        back_btn.pack(side="right", padx=20, pady=10)

        # Content frame
        content = tk.Frame(self.root, bg="#f7f7fa")
        content.pack(fill="both", expand=True, padx=20, pady=20)

        # Button grid
        buttons = [
            ("Request I/O", self.request_io_dialog),
            ("Complete I/O", self.release_io_dialog),
            ("View I/O Status", lambda: self.show_io_table(content))
        ]

        for i, (text, cmd) in enumerate(buttons):
            btn = tk.Button(content, text=text, command=cmd,
                           font=("Segoe UI", 11), bg="#23395d", fg="white",
                           height=2, width=20, wraplength=150)
            btn.grid(row=i, column=0, padx=5, pady=5, sticky="nsew")
            content.grid_columnconfigure(0, weight=1)

        for i in range(3):
            content.grid_rowconfigure(i, weight=1)

        # Show I/O table at bottom
        self.show_io_table(parent=content)

    def request_io_dialog(self):
        dialog = tk.Toplevel(self.root)
        dialog.title("Request I/O")
        dialog.configure(bg="#f7f7fa")

        tk.Label(dialog, text="Process ID:", font=("Segoe UI", 12), bg="#f7f7fa").pack(pady=10)
        pid_entry = tk.Entry(dialog, font=("Segoe UI", 12), width=20)
        pid_entry.pack(pady=5)

        def request():
            try:
                pid = int(pid_entry.get())
                if pid not in self.processes:
                    messagebox.showerror("Error", "Process not found")
                    return

                p = self.processes[pid]
                if p.io_state != "No I/O":
                    messagebox.showerror("Error", "Process already has I/O operation")
                    return

                p.io_state = "Waiting for I/O"
                p.state = "Blocked"
                self.blocked_queue.append(p)
                if p in self.ready_queue:
                    self.ready_queue.remove(p)
                if self.running_process and self.running_process.process_id == pid:
                    self.running_process = None
                    
                # Update process file
                self.create_process_file(p)
                
                dialog.destroy()
                self.show_io_management()
            except ValueError:
                messagebox.showerror("Error", "Invalid process ID")

        tk.Button(dialog, text="Request", command=request, font=("Segoe UI", 12, "bold"),
                 bg="#23395d", fg="#fff", width=15).pack(side="left", padx=10, pady=10)
        tk.Button(dialog, text="Cancel", command=dialog.destroy, font=("Segoe UI", 12, "bold"),
                 bg="#b7b1f3", fg="#fff", width=15).pack(side="right", padx=10, pady=10)

    def release_io_dialog(self):
        dialog = tk.Toplevel(self.root)
        dialog.title("Complete I/O")
        dialog.configure(bg="#f7f7fa")

        tk.Label(dialog, text="Process ID:", font=("Segoe UI", 12), bg="#f7f7fa").pack(pady=10)
        pid_entry = tk.Entry(dialog, font=("Segoe UI", 12), width=20)
        pid_entry.pack(pady=5)

        def release():
            try:
                pid = int(pid_entry.get())
                if pid not in self.processes:
                    messagebox.showerror("Error", "Process not found")
                    return

                p = self.processes[pid]
                if p.io_state == "No I/O":
                    messagebox.showerror("Error", "Process has no I/O operation")
                    return

                p.io_state = "No I/O"
                if p in self.blocked_queue:
                    self.blocked_queue.remove(p)
                p.state = "Ready"
                self.ready_queue.append(p)
                
                # Update process file
                self.create_process_file(p)
                
                dialog.destroy()
                self.show_io_management()
            except ValueError:
                messagebox.showerror("Error", "Invalid process ID")

        tk.Button(dialog, text="Complete", command=release, font=("Segoe UI", 12, "bold"),
                 bg="#23395d", fg="#fff", width=15).pack(side="left", padx=10, pady=10)
        tk.Button(dialog, text="Cancel", command=dialog.destroy, font=("Segoe UI", 12, "bold"),
                 bg="#b7b1f3", fg="#fff", width=15).pack(side="right", padx=10, pady=10)

    def show_io_table(self, parent):
        frame = tk.Frame(parent, bg="#f7f7fa")
        frame.grid(row=3, column=0, sticky="nsew", padx=10, pady=10)

        cols = ("ProcessID", "Name", "I/O State", "State")

        tree = ttk.Treeview(frame, columns=cols, show="headings", height=10)
        tree.heading("ProcessID", text="ProcessID")
        tree.heading("Name", text="Name")
        tree.heading("I/O State", text="I/O State")
        tree.heading("State", text="State")

        for p in self.processes.values():
            tree.insert("", "end", values=(
                p.process_id, p.name, p.io_state, p.state
            ))

        tree.pack(fill="both", expand=True)

    # -------------------- SYNCHRONIZATION OPERATIONS --------------------
    def show_sync_operations(self):
        self.clear_window()

        # Header
        header = tk.Frame(self.root, bg="#23395d")
        header.pack(fill="x")

        tk.Label(header, text="Synchronization Operations", font=("Segoe UI", 16, "bold"),
                bg="#23395d", fg="white").pack(side="left", padx=20, pady=10)

        back_btn = tk.Button(header, text="Back", font=("Segoe UI", 10),
                            bg="#b7b1f3", fg="#23395d", command=self.show_main_panel)
        back_btn.pack(side="right", padx=20, pady=10)

        # Content frame
        content = tk.Frame(self.root, bg="#f7f7fa")
        content.pack(fill="both", expand=True, padx=20, pady=20)

        # Synchronization frame
        sync_frame = tk.LabelFrame(content, text="Process Synchronization", font=("Segoe UI", 12, "bold"),
                                 bg="#f7f7fa", fg="#23395d")
        sync_frame.grid(row=0, column=0, padx=10, pady=10, sticky="nsew")

        sync_buttons = [
            ("Create Semaphore", self.create_semaphore_dialog),
            ("Semaphore Wait (P)", self.semaphore_wait_dialog),
            ("Semaphore Signal (V)", self.semaphore_signal_dialog),
            ("View Semaphores", self.view_semaphores_dialog),
            ("Request Resource", self.request_resource_dialog),
            ("Release Resource", self.release_resource_dialog),
        ]

        for i, (text, cmd) in enumerate(sync_buttons):
            btn = tk.Button(sync_frame, text=text, command=cmd,
                           font=("Segoe UI", 11), bg="#23395d", fg="white",
                           height=2, width=20, wraplength=150)
            btn.grid(row=i, column=0, padx=5, pady=5, sticky="nsew")
            sync_frame.grid_columnconfigure(0, weight=1)

        # IPC frame
        ipc_frame = tk.LabelFrame(content, text="Inter-process Communication", font=("Segoe UI", 12, "bold"),
                                bg="#f7f7fa", fg="#23395d")
        ipc_frame.grid(row=0, column=1, padx=10, pady=10, sticky="nsew")

        ipc_buttons = [
            ("Shared Memory", self.shared_memory_dialog),
            ("Message Passing", self.message_queue_dialog),
            ("Socket Communication", self.socket_client_dialog),
            ("Start Socket Server", self.socket_server_start_dialog)
        ]

        for i, (text, cmd) in enumerate(ipc_buttons):
            btn = tk.Button(ipc_frame, text=text, command=cmd,
                           font=("Segoe UI", 11), bg="#23395d", fg="white",
                           height=2, width=20, wraplength=150)
            btn.grid(row=i, column=0, padx=5, pady=5, sticky="nsew")
            ipc_frame.grid_columnconfigure(0, weight=1)

        # Configure grid weights
        content.grid_columnconfigure(0, weight=1)
        content.grid_columnconfigure(1, weight=1)
        content.grid_rowconfigure(0, weight=1)

    def create_semaphore_dialog(self):
        dialog = tk.Toplevel(self.root)
        dialog.title("Create Semaphore")
        dialog.configure(bg="#f7f7fa")

        tk.Label(dialog, text="Semaphore Name:", font=("Segoe UI", 12), bg="#f7f7fa").pack(pady=10)
        name_entry = tk.Entry(dialog, font=("Segoe UI", 12), width=20)
        name_entry.pack(pady=5)

        tk.Label(dialog, text="Initial Value:", font=("Segoe UI", 12), bg="#f7f7fa").pack(pady=10)
        value_entry = tk.Entry(dialog, font=("Segoe UI", 12), width=20)
        value_entry.insert(0, "1")
        value_entry.pack(pady=5)

        def create():
            name = name_entry.get().strip()
            if not name:
                messagebox.showerror("Error", "Name required")
                return

            try:
                value = int(value_entry.get())
                if value < 0:
                    raise ValueError()
            except ValueError:
                messagebox.showerror("Error", "Value must be non-negative integer")
                return

            if name in self.semaphores:
                messagebox.showerror("Error", "Semaphore already exists")
                return

            self.semaphores[name] = Semaphore(name, value)
            dialog.destroy()
            self.view_semaphores_dialog()

        tk.Button(dialog, text="Create", command=create, font=("Segoe UI", 12, "bold"),
                 bg="#23395d", fg="#fff", width=15).pack(side="left", padx=10, pady=10)
        tk.Button(dialog, text="Cancel", command=dialog.destroy, font=("Segoe UI", 12, "bold"),
                 bg="#b7b1f3", fg="#fff", width=15).pack(side="right", padx=10, pady=10)

    def semaphore_wait_dialog(self):
        dialog = tk.Toplevel(self.root)
        dialog.title("Semaphore Wait (P)")
        dialog.configure(bg="#f7f7fa")

        tk.Label(dialog, text="Semaphore Name:", font=("Segoe UI", 12), bg="#f7f7fa").pack(pady=10)
        sem_entry = tk.Entry(dialog, font=("Segoe UI", 12), width=20)
        sem_entry.pack(pady=5)

        tk.Label(dialog, text="Process ID:", font=("Segoe UI", 12), bg="#f7f7fa").pack(pady=10)
        pid_entry = tk.Entry(dialog, font=("Segoe UI", 12), width=20)
        pid_entry.pack(pady=5)

        def wait():
            name = sem_entry.get().strip()
            try:
                pid = int(pid_entry.get())
                if pid not in self.processes:
                    messagebox.showerror("Error", "Process not found")
                    return
            except ValueError:
                messagebox.showerror("Error", "Invalid process ID")
                return

            if name not in self.semaphores:
                messagebox.showerror("Error", "Semaphore not found")
                return

            sem = self.semaphores[name]
            if sem.value > 0:
                sem.value -= 1
                messagebox.showinfo("Success", f"Process {pid} acquired semaphore {name}")
            else:
                sem.queue.append(pid)
                self.processes[pid].state = "Blocked"
                if self.processes[pid] in self.ready_queue:
                    self.ready_queue.remove(self.processes[pid])
                messagebox.showinfo("Info", f"Process {pid} blocked on semaphore {name}")

            # Update process file
            self.create_process_file(self.processes[pid])
            
            dialog.destroy()
            self.view_semaphores_dialog()

        tk.Button(dialog, text="Wait (P)", command=wait, font=("Segoe UI", 12, "bold"),
                 bg="#23395d", fg="#fff", width=15).pack(side="left", padx=10, pady=10)
        tk.Button(dialog, text="Cancel", command=dialog.destroy, font=("Segoe UI", 12, "bold"),
                 bg="#b7b1f3", fg="#fff", width=15).pack(side="right", padx=10, pady=10)

    def semaphore_signal_dialog(self):
        dialog = tk.Toplevel(self.root)
        dialog.title("Semaphore Signal (V)")
        dialog.configure(bg="#f7f7fa")

        tk.Label(dialog, text="Semaphore Name:", font=("Segoe UI", 12), bg="#f7f7fa").pack(pady=10)
        sem_entry = tk.Entry(dialog, font=("Segoe UI", 12), width=20)
        sem_entry.pack(pady=5)

        tk.Label(dialog, text="Process ID:", font=("Segoe UI", 12), bg="#f7f7fa").pack(pady=10)
        pid_entry = tk.Entry(dialog, font=("Segoe UI", 12), width=20)
        pid_entry.pack(pady=5)

        def signal():
            name = sem_entry.get().strip()
            try:
                pid = int(pid_entry.get())
                if pid not in self.processes:
                    messagebox.showerror("Error", "Process not found")
                    return
            except ValueError:
                messagebox.showerror("Error", "Invalid process ID")
                return

            if name not in self.semaphores:
                messagebox.showerror("Error", "Semaphore not found")
                return

            sem = self.semaphores[name]
            
            if sem.queue:
                released_pid = sem.queue.popleft()
                self.processes[released_pid].state = "Ready"
                self.ready_queue.append(self.processes[released_pid])
                messagebox.showinfo("Success", f"Process {released_pid} released from semaphore {name}")
                
                # Update process file
                self.create_process_file(self.processes[released_pid])
            else:
                sem.value += 1
                messagebox.showinfo("Success", f"Semaphore {name} value incremented to {sem.value}")

            dialog.destroy()
            self.view_semaphores_dialog()

        tk.Button(dialog, text="Signal (V)", command=signal, font=("Segoe UI", 12, "bold"),
                 bg="#23395d", fg="#fff", width=15).pack(side="left", padx=10, pady=10)
        tk.Button(dialog, text="Cancel", command=dialog.destroy, font=("Segoe UI", 12, "bold"),
                 bg="#b7b1f3", fg="#fff", width=15).pack(side="right", padx=10, pady=10)

    def view_semaphores_dialog(self):
        dialog = tk.Toplevel(self.root)
        dialog.title("Semaphores")
        dialog.configure(bg="#f7f7fa")

        if not self.semaphores:
            tk.Label(dialog, text="No semaphores created", font=("Segoe UI", 12), bg="#f7f7fa").pack(pady=20)
        else:
            frame = tk.Frame(dialog, bg="#f7f7fa")
            frame.pack(fill="both", expand=True, padx=10, pady=10)

            cols = ("Name", "Value", "Queue")
            tree = ttk.Treeview(frame, columns=cols, show="headings", height=10)

            for col in cols:
                tree.heading(col, text=col)
                tree.column(col, width=150, anchor=tk.CENTER)

            for sem in self.semaphores.values():
                tree.insert("", "end", values=(sem.name, sem.value, list(sem.queue)))

            tree.pack(fill="both", expand=True)

        tk.Button(dialog, text="Close", command=dialog.destroy, font=("Segoe UI", 12, "bold"),
                 bg="#b7b1f3", fg="#fff", width=15).pack(pady=10)

    def request_resource_dialog(self):
        dialog = tk.Toplevel(self.root)
        dialog.title("Request Resource")
        dialog.configure(bg="#f7f7fa")

        tk.Label(dialog, text="Process ID:", font=("Segoe UI", 12), bg="#f7f7fa").pack(pady=10)
        pid_entry = tk.Entry(dialog, font=("Segoe UI", 12), width=20)
        pid_entry.pack(pady=5)

        tk.Label(dialog, text="Resource Type:", font=("Segoe UI", 12), bg="#f7f7fa").pack(pady=10)
        resource_var = tk.StringVar(value="printer")
        tk.OptionMenu(dialog, resource_var, *self.resources.keys()).pack()

        def request():
            try:
                pid = int(pid_entry.get())
                if pid not in self.processes:
                    messagebox.showerror("Error", "Process not found")
                    return
                    
                res_type = resource_var.get()
                res = self.resources[res_type]
                
                if len(res.allocated) < res.total:
                    res.allocated[pid] = True
                    messagebox.showinfo("Success", f"Resource {res_type} allocated to Process {pid}")
                else:
                    messagebox.showwarning("Wait", 
                        f"Process {pid} must wait for {res_type} (all instances in use)")
                    
                    # Block the process
                    self.processes[pid].state = "Blocked"
                    self.blocked_queue.append(self.processes[pid])
            except ValueError:
                messagebox.showerror("Error", "Invalid input")
        
        tk.Button(dialog, text="Request", command=request, font=("Segoe UI", 12, "bold"),
                 bg="#23395d", fg="#fff", width=15).pack(side="left", padx=10, pady=10)
        tk.Button(dialog, text="Cancel", command=dialog.destroy, font=("Segoe UI", 12, "bold"),
                 bg="#b7b1f3", fg="#fff", width=15).pack(side="right", padx=10, pady=10)

    def release_resource_dialog(self):
        dialog = tk.Toplevel(self.root)
        dialog.title("Release Resource")
        dialog.configure(bg="#f7f7fa")

        tk.Label(dialog, text="Process ID:", font=("Segoe UI", 12), bg="#f7f7fa").pack(pady=10)
        pid_entry = tk.Entry(dialog, font=("Segoe UI", 12), width=20)
        pid_entry.pack(pady=5)

        tk.Label(dialog, text="Resource Type:", font=("Segoe UI", 12), bg="#f7f7fa").pack(pady=10)
        resource_var = tk.StringVar(value="printer")
        tk.OptionMenu(dialog, resource_var, *self.resources.keys()).pack()

        def release():
            try:
                pid = int(pid_entry.get())
                res_type = resource_var.get()
                
                if pid in self.resources[res_type].allocated:
                    del self.resources[res_type].allocated[pid]
                    messagebox.showinfo("Success", f"Resource {res_type} released by Process {pid}")
                else:
                    messagebox.showerror("Error", "Process doesn't hold this resource")
            except ValueError:
                messagebox.showerror("Error", "Invalid input")
        
        tk.Button(dialog, text="Release", command=release, font=("Segoe UI", 12, "bold"),
                 bg="#23395d", fg="#fff", width=15).pack(side="left", padx=10, pady=10)
        tk.Button(dialog, text="Cancel", command=dialog.destroy, font=("Segoe UI", 12, "bold"),
                 bg="#b7b1f3", fg="#fff", width=15).pack(side="right", padx=10, pady=10)

    def shared_memory_dialog(self):
        dialog = tk.Toplevel(self.root)
        dialog.title("Shared Memory")
        dialog.configure(bg="#f7f7fa")

        tk.Label(dialog, text="Shared Memory Segments:", font=("Segoe UI", 12, "bold"), bg="#f7f7fa").pack(pady=10)

        if not self.shared_memory:
            tk.Label(dialog, text="No shared memory segments", font=("Segoe UI", 12), bg="#f7f7fa").pack()
        else:
            frame = tk.Frame(dialog, bg="#f7f7fa")
            frame.pack(fill="both", expand=True, padx=10, pady=5)

            text = tk.Text(frame, height=10, width=50, font=("Segoe UI", 10))
            scroll = tk.Scrollbar(frame, command=text.yview)
            text.configure(yscrollcommand=scroll.set)

            for name, value in self.shared_memory.items():
                text.insert(tk.END, f"{name}: {value}\n")

            text.pack(side="left", fill="both", expand=True)
            scroll.pack(side="right", fill="y")

        tk.Label(dialog, text="Segment Name:", font=("Segoe UI", 12), bg="#f7f7fa").pack(pady=5)
        name_entry = tk.Entry(dialog, font=("Segoe UI", 12), width=20)
        name_entry.pack(pady=5)

        tk.Label(dialog, text="Value:", font=("Segoe UI", 12), bg="#f7f7fa").pack(pady=5)
        value_entry = tk.Entry(dialog, font=("Segoe UI", 12), width=20)
        value_entry.pack(pady=5)

        def set_segment():
            name = name_entry.get().strip()
            value = value_entry.get().strip()

            if not name:
                messagebox.showerror("Error", "Name required")
                return

            self.shared_memory[name] = value
            dialog.destroy()
            self.shared_memory_dialog()

        def delete_segment():
            name = name_entry.get().strip()

            if name not in self.shared_memory:
                messagebox.showerror("Error", "Segment not found")
                return

            del self.shared_memory[name]
            dialog.destroy()
            self.shared_memory_dialog()

        btn_frame = tk.Frame(dialog, bg="#f7f7fa")
        btn_frame.pack(pady=10)

        tk.Button(btn_frame, text="Set", command=set_segment, font=("Segoe UI", 12, "bold"),
                 bg="#23395d", fg="#fff", width=10).pack(side="left", padx=5)
        tk.Button(btn_frame, text="Delete", command=delete_segment, font=("Segoe UI", 12, "bold"),
                 bg="#e0c3fc", fg="#23395d", width=10).pack(side="left", padx=5)
        tk.Button(dialog, text="Close", command=dialog.destroy, font=("Segoe UI", 12, "bold"),
                 bg="#b7b1f3", fg="#fff", width=15).pack(pady=10)

    def message_queue_dialog(self):
        dialog = tk.Toplevel(self.root)
        dialog.title("Message Queues")
        dialog.configure(bg="#f7f7fa")

        tk.Label(dialog, text="Message Queues:", font=("Segoe UI", 12, "bold"), bg="#f7f7fa").pack(pady=10)

        if not self.message_queues:
            tk.Label(dialog, text="No message queues", font=("Segoe UI", 12), bg="#f7f7fa").pack()
        else:
            frame = tk.Frame(dialog, bg="#f7f7fa")
            frame.pack(fill="both", expand=True, padx=10, pady=5)

            text = tk.Text(frame, height=10, width=50, font=("Segoe UI", 10))
            scroll = tk.Scrollbar(frame, command=text.yview)
            text.configure(yscrollcommand=scroll.set)

            for qname, queue in self.message_queues.items():
                messages = list(queue.queue)
                if messages:
                    text.insert(tk.END, f"Queue for PID {qname}:\n")
                    for msg in messages:
                        text.insert(tk.END, f"  From PID {msg['sender']}: {msg['message']}\n")
                    text.insert(tk.END, "\n")

            text.pack(side="left", fill="both", expand=True)
            scroll.pack(side="right", fill="y")

        tk.Label(dialog, text="Receiver PID:", font=("Segoe UI", 12), bg="#f7f7fa").pack(pady=5)
        receiver_entry = tk.Entry(dialog, font=("Segoe UI", 12), width=20)
        receiver_entry.pack(pady=5)

        tk.Label(dialog, text="Message:", font=("Segoe UI", 12), bg="#f7f7fa").pack(pady=5)
        msg_entry = tk.Entry(dialog, font=("Segoe UI", 12), width=20)
        msg_entry.pack(pady=5)

        def send_message():
            try:
                sender_pid = simpledialog.askinteger("Sender PID", "Enter your process ID:")
                if sender_pid is None or sender_pid not in self.processes:
                    messagebox.showerror("Error", "Invalid sender process ID")
                    return

                receiver_pid = int(receiver_entry.get())
                if receiver_pid not in self.processes:
                    messagebox.showerror("Error", "Invalid receiver process ID")
                    return

                msg = msg_entry.get().strip()
                if not msg:
                    messagebox.showerror("Error", "Message required")
                    return

                # Store message with sender info
                self.message_queues[receiver_pid].put({
                    "sender": sender_pid,
                    "message": msg,
                    "timestamp": datetime.now().isoformat()
                })
                dialog.destroy()
                self.message_queue_dialog()
            except ValueError:
                messagebox.showerror("Error", "Invalid PID")

        def receive_message():
            try:
                receiver_pid = int(receiver_entry.get())
                if receiver_pid not in self.processes:
                    messagebox.showerror("Error", "Invalid process ID")
                    return

                if receiver_pid not in self.message_queues or self.message_queues[receiver_pid].empty():
                    messagebox.showerror("Error", "No messages for this process")
                    return

                msg_data = self.message_queues[receiver_pid].get()
                messagebox.showinfo("Message Received", 
                                  f"From PID {msg_data['sender']} at {msg_data['timestamp']}:\n"
                                  f"{msg_data['message']}")
                dialog.destroy()
                self.message_queue_dialog()
            except ValueError:
                messagebox.showerror("Error", "Invalid PID")

        btn_frame = tk.Frame(dialog, bg="#f7f7fa")
        btn_frame.pack(pady=10)

        tk.Button(btn_frame, text="Send", command=send_message, font=("Segoe UI", 12, "bold"),
                 bg="#23395d", fg="#fff", width=10).pack(side="left", padx=5)
        tk.Button(btn_frame, text="Receive", command=receive_message, font=("Segoe UI", 12, "bold"),
                 bg="#e0c3fc", fg="#23395d", width=10).pack(side="left", padx=5)
        tk.Button(dialog, text="Close", command=dialog.destroy, font=("Segoe UI", 12, "bold"),
                 bg="#b7b1f3", fg="#fff", width=15).pack(pady=10)

    def socket_server_start_dialog(self):
        if self.socket_server_thread and self.socket_server_thread.is_alive():
            messagebox.showinfo("Socket Server", "Server is already running")
            return

        port = simpledialog.askinteger("Socket Server", "Enter port (default 9000):", initialvalue=9000)
        if not port:
            return

        self.socket_server_thread = SocketServerThread(self, port)
        self.socket_server_thread.start()
        messagebox.showinfo("Socket Server", f"Server started on port {port}")

    def socket_client_dialog(self):
        dialog = tk.Toplevel(self.root)
        dialog.title("Socket Client")
        dialog.configure(bg="#f7f7fa")

        tk.Label(dialog, text="Server Host:", font=("Segoe UI", 12), bg="#f7f7fa").pack(pady=5)
        host_entry = tk.Entry(dialog, font=("Segoe UI", 12), width=20)
        host_entry.insert(0, "localhost")
        host_entry.pack(pady=5)

        tk.Label(dialog, text="Port:", font=("Segoe UI", 12), bg="#f7f7fa").pack(pady=5)
        port_entry = tk.Entry(dialog, font=("Segoe UI", 12), width=20)
        port_entry.insert(0, "9000")
        port_entry.pack(pady=5)

        tk.Label(dialog, text="Message:", font=("Segoe UI", 12), bg="#f7f7fa").pack(pady=5)
        msg_entry = tk.Entry(dialog, font=("Segoe UI", 12), width=20)
        msg_entry.pack(pady=5)

        def send():
            host = host_entry.get().strip()
            try:
                port = int(port_entry.get())
            except ValueError:
                messagebox.showerror("Error", "Invalid port number")
                return

            msg = msg_entry.get().strip()
            if not msg:
                messagebox.showerror("Error", "Message required")
                return

            try:
                s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
                s.connect((host, port))
                s.sendall(msg.encode())
                response = s.recv(1024).decode()
                s.close()
                messagebox.showinfo("Response", f"Server response:\n{response}")
            except Exception as e:
                messagebox.showerror("Error", f"Connection failed: {e}")

        tk.Button(dialog, text="Send", command=send, font=("Segoe UI", 12, "bold"),
                 bg="#23395d", fg="#fff", width=15).pack(side="left", padx=10, pady=10)
        tk.Button(dialog, text="Cancel", command=dialog.destroy, font=("Segoe UI", 12, "bold"),
                 bg="#b7b1f3", fg="#fff", width=15).pack(side="right", padx=10, pady=10)

    # -------------------- CONFIGURATION --------------------
    def show_configuration(self):
        self.clear_window()
        card = tk.Frame(self.root, bg="#f7f7fa", bd=2, relief=tk.RIDGE)
        card.place(relx=0.5, rely=0.5, anchor="center", relwidth=0.6, relheight=0.9)

        title = tk.Label(card, text="Configuration", font=("Segoe UI", 20, "bold"),
                        bg="#f7f7fa", fg="#23395d")
        title.pack(pady=10)

        # Scheduling algorithm
        sched_frame = tk.Frame(card, bg="#f7f7fa")
        sched_frame.pack(pady=10)
        tk.Label(sched_frame, text="Scheduling Algorithm:", font=("Segoe UI", 12), bg="#f7f7fa").pack()

        self.sched_var = tk.StringVar(value=self.scheduling_algorithm)
        for algo in self.scheduling_algorithms:
            tk.Radiobutton(sched_frame, text=algo, variable=self.sched_var, value=algo,
                          font=("Segoe UI", 11), bg="#f7f7fa").pack(anchor="w")

        # Time quantum for RR
        quantum_frame = tk.Frame(card, bg="#f7f7fa")
        quantum_frame.pack(pady=10)
        tk.Label(quantum_frame, text="Time Quantum (for RR):", font=("Segoe UI", 12), bg="#f7f7fa").pack(side="left")
        self.quantum_entry = tk.Entry(quantum_frame, font=("Segoe UI", 12), width=10)
        self.quantum_entry.insert(0, str(self.time_quantum))
        self.quantum_entry.pack(side="left", padx=5)

        # Memory configuration
        mem_frame = tk.Frame(card, bg="#f7f7fa")
        mem_frame.pack(pady=10)
        tk.Label(mem_frame, text="Page Size (KB):", font=("Segoe UI", 12), bg="#f7f7fa").pack(side="left")
        self.page_size_entry = tk.Entry(mem_frame, font=("Segoe UI", 12), width=10)
        self.page_size_entry.insert(0, str(PAGE_SIZE // 1024))
        self.page_size_entry.pack(side="left", padx=5)

        # Apply button
        tk.Button(card, text="Apply Configuration", command=self.apply_configuration,
                 font=("Segoe UI", 12, "bold"), bg="#23395d", fg="#fff", width=20, pady=6).pack(pady=10)

        # Back button
        tk.Button(card, text="Back", command=self.show_main_panel, font=("Segoe UI", 12, "bold"),
                 bg="#b7b1f3", fg="#fff", width=20, pady=6).pack()

    def apply_configuration(self):
        # Update scheduling algorithm
        self.scheduling_algorithm = self.sched_var.get()

        # Update time quantum
        try:
            new_quantum = int(self.quantum_entry.get())
            if new_quantum <= 0:
                raise ValueError()
            self.time_quantum = new_quantum
        except:
            messagebox.showerror("Error", "Invalid time quantum (must be positive integer)")
            return

        messagebox.showinfo("Success", f"Configuration applied:\n"
                                      f"Scheduling: {self.scheduling_algorithm}\n"
                                      f"Time Quantum: {self.time_quantum}\n"
                                      f"Page Size: {PAGE_SIZE // 1024}KB")

if __name__ == "__main__":
    root = tk.Tk()
    app = OSSimulationApp(root)
    root.mainloop()